# Vital signs — data analysis & models

Run cells **top to bottom**. Works if your notebook kernel’s **current working directory** is `model/` *or* the **repo root** (the next cell adds the `model` folder to `sys.path`).

**Install (once):**
```bash
cd model
pip install -r requirements.txt
pip install jupyter ipykernel
```

Then start Jupyter from `model/`, or open this file in Cursor/VS Code and select a Python interpreter.

In [15]:
# Optional: install deps inside the notebook
%pip install -q numpy pandas scikit-learn

import sys
from pathlib import Path

HERE = Path.cwd().resolve()
MODEL_DIR = HERE / "model" if (HERE / "model" / "health_rules.py").exists() else HERE
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

print("Using model code from:", MODEL_DIR)

import pandas as pd
from health_rules import rule_status_series
from preprocessing import clean_frame, one_hot_status
from augmentation import bootstrap_expand
from analysis_stats import numeric_summary, status_value_counts, group_stats_by_status, correlation_matrix
from ml_train import train_and_evaluate
from sample_data import ensure_sample_csv

Note: you may need to restart the kernel to use updated packages.
Using model code from: D:\2026_PROJECT\vital sign\vital-sign-v2\model



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Load data

The loader now prefers your downloaded full export (`vitals_export_*.csv`) from `model/data/`.

Priority order:
1) Newest `vitals_export_*.csv`
2) `vitals.csv`
3) Auto-generate synthetic sample

In [9]:
DATA_DIR = MODEL_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

export_candidates = sorted(DATA_DIR.glob("vitals_export_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if export_candidates:
    DATA = export_candidates[0]
    print("Using newest export:", DATA.name)
else:
    DATA = DATA_DIR / "vitals.csv"
    if not DATA.exists():
        ensure_sample_csv(DATA, n=500)
        print("Created sample:", DATA)
    else:
        print("Using base dataset:", DATA.name)

df = pd.read_csv(DATA)
df["rule_status"] = rule_status_series(df)
print(f"Loaded {len(df):,} rows from {DATA.name}")
df.head()

,id,temperature,heart_rate,spo2,status,recommendation,created_at,rule_status
0,1773510,36.29,113,98,WARNING,Rest and monitor your condition,2026-04-23T08:04:57.538516+00:00,WARNING
1,7850256,36.04,84,98,SAFE,You are in good health,2026-04-23T08:04:57.543698+00:00,SAFE
2,5503166,36.59,121,96,WARNING,Rest and monitor your condition,2026-04-23T08:04:57.543698+00:00,WARNING
3,4990727,37.25,76,90,ALERT,Visit the clinic immediately,2026-04-23T08:04:57.545111+00:00,ALERT
4,8724620,36.63,91,95,SAFE,You are in good health,2026-04-23T08:04:57.545111+00:00,SAFE


## 2. Cleaning & quality check

In [10]:
if "status" in df.columns:
    agree = (df["status"].astype(str) == df["rule_status"].astype(str)).mean()
    print(f"Stored status vs rule_status agreement: {agree:.3%}")

cleaned, report = clean_frame(df, drop_iqr_outliers=False, drop_clinical_extreme=True)
report

Stored status vs rule_status agreement: 100.000%


{'initial_rows': 500,
 'rows_after_drop_na': 500,
 'removed_invalid': 0,
 'clinical_extreme_count': 0,
 'final_rows': 500}

## 3. Bootstrap augmentation + summary statistics

In [11]:
aug_n = max(len(cleaned) * 3, len(cleaned) + 1)
augmented = bootstrap_expand(cleaned, n_rows=aug_n, random_state=42)
numeric_summary(augmented)

,temperature,heart_rate,spo2
count,1500.000000,1500.000000,1500.000000
mean,36.754487,84.780000,95.772000
std,0.741593,17.328968,2.846299
min,35.470000,60.000000,88.000000
25%,36.340000,71.000000,95.000000
50%,36.590000,82.000000,96.000000
75%,36.860000,93.000000,98.000000
max,39.350000,124.000000,99.000000


## 4. Groups & correlation

In [12]:
display(status_value_counts(cleaned, "rule_status"))
display(group_stats_by_status(cleaned, "rule_status"))
correlation_matrix(cleaned)

rule_status
SAFE       337
WARNING     89
ALERT       74
Name: count, dtype: int64

temperature                  heart_rate                   \
                   mean       std count        mean        std count   
rule_status                                                            
ALERT         38.060000  1.003843    74   85.675676  10.837667    74   
SAFE          36.493501  0.348898   337   76.801187  10.363581   337   
WARNING       36.637303  0.318868    89  114.382022   5.808508    89   

                  spo2                  
                  mean       std count  
rule_status                             
ALERT        89.905405  1.406140    74  
SAFE         97.029674  1.409685   337  
WARNING      96.123596  1.355295    89

,temperature,heart_rate,spo2
temperature,1.000000,0.105246,-0.632544
heart_rate,0.105246,1.000000,-0.116891
spo2,-0.632544,-0.116891,1.000000


## 5. One-hot encoding (example)

In [13]:
oh, enc = one_hot_status(cleaned["rule_status"].head(10))
oh

,status_ALERT,status_SAFE,status_WARNING
0,0.0,0.0,1.0
1,0.0,1.0,0.0
2,0.0,0.0,1.0
3,1.0,0.0,0.0
4,0.0,1.0,0.0
5,0.0,1.0,0.0
6,1.0,0.0,0.0
7,0.0,1.0,0.0
8,0.0,0.0,1.0
9,0.0,1.0,0.0


## 6. Train & compare models

Needs enough rows after cleaning (≥ ~15).

In [14]:
if len(cleaned) < 15:
    print("Not enough rows; increase sample size or add real data.")
else:
    out = train_and_evaluate(cleaned, target_col="rule_status", random_state=42)
    for name, payload in out["results"].items():
        print(f"\n=== {name} — accuracy: {payload['accuracy']:.4f} ===")
        print(payload["report"])


=== logistic_regression — accuracy: 1.0000 ===
              precision    recall  f1-score   support

       ALERT       1.00      1.00      1.00        19
        SAFE       1.00      1.00      1.00        84
     WARNING       1.00      1.00      1.00        22

    accuracy                           1.00       125
   macro avg       1.00      1.00      1.00       125
weighted avg       1.00      1.00      1.00       125


=== random_forest — accuracy: 1.0000 ===
              precision    recall  f1-score   support

       ALERT       1.00      1.00      1.00        19
        SAFE       1.00      1.00      1.00        84
     WARNING       1.00      1.00      1.00        22

    accuracy                           1.00       125
   macro avg       1.00      1.00      1.00       125
weighted avg       1.00      1.00      1.00       125


=== knn_k5 — accuracy: 1.0000 ===
              precision    recall  f1-score   support

       ALERT       1.00      1.00      1.00        19
    